# Build the tillage montages (Google Colab)

Python in the browser — nothing to install locally.

**This is NOT the Earth Engine Code Editor.** The Code Editor only runs JavaScript and has no
filesystem, so it cannot stitch montages, draw the delineation, or write CSVs. That work is Python.
(The Code Editor file is `gee/prototype.js`, for eyeballing the imagery only.)

Run the cells top to bottom. Total ~20–40 min for 500 fields x 8 dates.

## 1. Install the bits Colab doesn't already have

In [ ]:
!pip install -q earthengine-api pyproj pillow requests
print('deps ready')

## 2. Authenticate to Earth Engine
Opens a Google sign-in. `EE_PROJECT` must be a Cloud project with the Earth Engine API enabled
(`ca-morocco` came from your `gcloud projects list` — change it if that's the wrong one).

In [ ]:
import ee

EE_PROJECT = 'ca-morocco'

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print(ee.String('GEE works — project: ' + EE_PROJECT).getInfo())

## 3. Pull the script + the 500 sampled fields
Straight from the public repo — no uploads. (We fetch just these two files rather than cloning,
because the repo still carries the old placeholder images.)

In [ ]:
RAW = 'https://raw.githubusercontent.com/Floriandbl/notill-validation/main'
!mkdir -p gee prep
!wget -q -O gee/build_from_gee.py {RAW}/gee/build_from_gee.py
!wget -q -O prep/fields_settat_500.geojson {RAW}/prep/fields_settat_500.geojson

import json
print('fields:', len(json.load(open('prep/fields_settat_500.geojson'))['features']))
!grep -E '^(EE_PROJECT|SEASON |BOX_M|N_STEPS|STEP_DAYS|PANEL_PX|CLEAN_FIRST)' gee/build_from_gee.py

## 4. (optional) Change settings without hand-editing
Uncomment a line to override. Reminder: the field is ~9 Sentinel-2 pixels **whatever** the box —
a bigger box only shrinks the field on screen and adds context.

In [ ]:
# !sed -i 's/^BOX_M     = 750/BOX_M     = 250/' gee/build_from_gee.py   # tighter framing
# !sed -i "s/^SEASON   = 2025/SEASON   = 2024/" gee/build_from_gee.py  # different season
!grep -E '^(SEASON |BOX_M)' gee/build_from_gee.py

## 5. Smoke-test on 3 fields FIRST
Never spend 4,000 downloads before looking at one. This writes 3 montages and shows one.

In [ ]:
import sys, json, importlib
sys.path.insert(0, 'gee')
import build_from_gee as B
importlib.reload(B)

feats = json.load(open(B.GEOJSON))['features'][:3]
windows = B.season_windows(B.SEASON)
for f in feats:
    rel, lon, lat = B.process(f, windows)
    print('wrote', rel)

from IPython.display import Image as Show
import glob
Show(sorted(glob.glob('images/**/*.jpg', recursive=True))[0])

**Look at that montage before continuing.** Is the delineation on the right field? Is the field big
enough to judge? Can you see it change across A..H? If not, fix `BOX_M` / `SEASON` / `VIS` in cell 4
and re-run — that is much cheaper than after 500 fields.

## 6. The full run (500 fields x 8 dates)

In [ ]:
!python gee/build_from_gee.py

## 7. Download the results, then commit them to the repo locally

In [ ]:
!zip -qr montages.zip images pairs_metadata.csv
!du -h montages.zip
from google.colab import files
files.download('montages.zip')

Then, on your machine: unzip into the App folder (replacing `images/` and `pairs_metadata.csv`),
run `Rscript r/build_pairs.R`, load `pairs_seed.sql` into Supabase, and `git push`.